# A18 · 广告请求到结算的端到端账本

先修：公用31–34；A17。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

串联产品入口→关键词触发→资格→质量排序→支付阈值→事件→账本。这个综合例使用CPC和构造点击，媒体收入等于广告主支付（无手续费）。报价只是上限，不等于支付；预算不足则本教学例不展示，生产系统应在服务前进行可支付资格或预留。真实Criteo CTR证据与这里的构造质量/事件严格分层，不能输出真实收入。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
served=ac.ledger(budget=2.);blocked=ac.ledger(budget=1.);empty=ac.ledger(query='未知',budget=2.)
print('成交',served);print('预算不足',blocked);print('无候选',empty)
assert served['spend']==served['publisher_income']>0
assert blocked['spend']==0 and empty['spend']==0
assert len({x['request'] for x in served['ledger']})==len(served['ledger'])
dedup=ac.deduplicate_charges([('click-1',.5),('click-1',.5),('click-2',.8)],1.)
print('重复事件及预算拒绝',dedup)
assert dedup['spend']==.5 and len(dedup['entries'])==1
print('原正式预测与模拟市场保持独立：参阅19–24与原API')

成交 {'candidates': ['a', 'b'], 'ledger': [{'request': 'constructed-1', 'ad': 'a', 'event': 'click', 'payment': 1.5625, 'receipt': 1.5625}], 'spend': 1.5625, 'publisher_income': 1.5625, 'probability_source': 'constructed quality, NOT Criteo production CTR', 'served': True}
预算不足 {'candidates': ['a', 'b'], 'ledger': [], 'spend': 0.0, 'publisher_income': 0.0, 'probability_source': 'constructed quality, NOT Criteo production CTR', 'served': False}
无候选 {'candidates': [], 'ledger': [], 'spend': 0.0, 'publisher_income': 0.0}
重复事件及预算拒绝 {'spend': 0.5, 'entries': [{'event': 'click-1', 'advertiser_debit': 0.5, 'publisher_credit': 0.5}], 'unique_events': 2}
原正式预测与模拟市场保持独立：参阅19–24与原API


## 解释与局限

成功、预算不足与空候选都需解释且不产生虚假扣费。此例只有一个请求，无并发、重试、退款或日志平台；本例已加入单进程事件ID幂等；下一步需实现并发原子预算与幂等持久化，不可以简单多次累加同一notice。

## 查看源码，再做练习

改变重复事件顺序并验证仍只扣费一次；再设计退款事件并检查广告主净支付=媒体净收款。

In [3]:
import inspect
print(inspect.getsource(ac.ledger))

def ledger(query='跑鞋',region='上海',budget=1.):
    pool=candidates(query,region); rows=[]; spent=0.
    if not pool: return dict(candidates=[],ledger=[],spend=0.,publisher_income=0.)
    winner=pool[0]; next_score=pool[1]['bid']*pool[1]['quality'] if len(pool)>1 else 0.
    cpc=next_score/winner['quality']
    # One explicitly constructed click; affordable eligibility is checked again at charging.
    click=True;payment=cpc if click and cpc<=budget else 0.
    if payment: spent+=payment; rows.append(dict(request='constructed-1',ad=winner['id'],event='click',payment=payment,receipt=payment))
    assert spent<=budget and abs(sum(r['payment'] for r in rows)-sum(r['receipt'] for r in rows))<1e-12
    return dict(candidates=[a['id'] for a in pool],ledger=rows,spend=spent,publisher_income=spent,
                probability_source='constructed quality, NOT Criteo production CTR',served=bool(rows))



## 来源与下一步

Lab自编综合，串联产品、候选、决策与账本。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。